# Box Office Bomb Data Pipeline Assignment

**Total Marks: 20**

This notebook implements an end-to-end pipeline to:
1. Scrape box office bomb data from Wikipedia
2. Validate & clean data using Pydantic
3. Enrich with OMDb API data
4. Perform consistency checks
5. Create final categorized dataset

## Setup and Imports

In [ ]:
# Import required libraries
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, field_validator, ValidationError
from typing import Optional
import re
import time
from pathlib import Path

## Task 1: Scrape the "Bombs" Table (4 Marks)

Extract raw data from the Wikipedia HTML file:
- Film Title (with symbols like § and †)
- Year
- Net production budget (may contain ranges like "$100–160")
- Estimated loss (Nominal column)

Note:
- You need to extract the entire raw string with the symbols, references, etc along with the titles.
- You must handle the nested headers in the Wikipedia table (Budget and Loss columns have sub-headers).

In [ ]:
def scrape_box_office_bombs():
    url = "https://en.wikipedia.org/wiki/List_of_biggest_box-office_bombs"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.select_one("table.wikitable.sortable.plainrowheaders")

    rows = table.select("tr")[2:]
    data = []

    for row in rows:
        title_cell = row.select_one("th")
        cells = row.select("td")

        data.append({
            "raw_title": title_cell.text.strip(),
            "raw_year": cells[0].text.strip(),
            "raw_budget": cells[1].text.strip(),
            "raw_loss": cells[2].text.strip()
        })

    return data



In [ ]:
# Test the scraping function
raw_movies = scrape_box_office_bombs()
print(f"Scraped {len(raw_movies)} movies")
print("\nLast 15 raw entries:")
for i, movie in enumerate(raw_movies[-15:]):
    print(f"\n{i+1}. {movie}")

Scraped 139 movies

Last 15 raw entries:

1. {'raw_title': 'Town & Country', 'raw_year': '2001', 'raw_budget': '$90', 'raw_loss': '$10.4'}

2. {'raw_title': 'Transformers: The Last Knight', 'raw_year': '2017', 'raw_budget': '$217–260', 'raw_loss': '$605.4'}

3. {'raw_title': 'Treasure Planet', 'raw_year': '2002', 'raw_budget': '$140', 'raw_loss': '$109.6'}

4. {'raw_title': 'Tron: Ares †', 'raw_year': '2025', 'raw_budget': '$220', 'raw_loss': '$142.2'}

5. {'raw_title': 'Turning Red §', 'raw_year': '2022', 'raw_budget': '$175', 'raw_loss': '$19.8'}

6. {'raw_title': 'Valerian and the City of a Thousand Planets', 'raw_year': '2017', 'raw_budget': '$177.2–180', 'raw_loss': '$225.9'}

7. {'raw_title': 'West Side Story', 'raw_year': '2021', 'raw_budget': '$100', 'raw_loss': '$76'}

8. {'raw_title': 'Wild Wild West', 'raw_year': '1999', 'raw_budget': '$170', 'raw_loss': '$217.7'}

9. {'raw_title': 'Windtalkers', 'raw_year': '2002', 'raw_budget': '$115–120', 'raw_loss': '$77.6'}

10. {'raw_t

## Task 2: Pydantic Data Parsing & Validation (6 Marks)

Create a Pydantic model that:
- Cleans titles (removes §, †, and footnotes like [nb 2], [1])
- Parses numeric values (handles ranges, currency symbols)
- Validates year as integer

In [ ]:
class MovieData(BaseModel):
    title: str
    year: int
    budget_millions: float
    loss_millions: float

    @field_validator("title", mode="before")
    @classmethod
    def clean_title(cls, v):
        v= v.replace("§", "").replace("†", "")
        v = re.sub(r"\[.*?\]", "", v)

        return v.strip()

    @field_validator("year", mode="before")
    @classmethod
    def validate_year(cls, v):
        year_str = re.sub(r"\D", "", str(v))
        if not year_str:
            raise ValueError("Invalid year")
        return int(year_str)

    @field_validator("budget_millions", mode="before")
    @classmethod
    def parse_budget(cls, v):
        return cls._parse_numeric_value(v)

    @field_validator("loss_millions", mode="before")
    @classmethod
    def parse_loss(cls, v):
        return cls._parse_numeric_value(v)

    @staticmethod
    def _parse_numeric_value(v):
        text = re.sub(r"\[.*?\]", "", str(v))
        text = text.replace("$", "").replace(",", "").strip()

        if "–" in text or "-" in text:
            parts = re.split(r"[–-]", text)
            nums = [float(p) for p in parts if re.match(r"\d+(\.\d+)?", p.strip())]
            if len(nums) == 2:
                return sum(nums) / 2

        match = re.search(r"\d+(\.\d+)?", text)
        if not match:
            raise ValueError("Unparsable numeric value")

        return float(match.group())


In [ ]:
# Validate and clean the raw data
validated_movies = []
failed_validations = []

for raw_movie in raw_movies:
    try:
        movie = MovieData(
            title=raw_movie['raw_title'],
            year=raw_movie['raw_year'],
            budget_millions=raw_movie['raw_budget'],
            loss_millions=raw_movie['raw_loss']
        )
        validated_movies.append(movie)
    except ValidationError as e:
        failed_validations.append({
            'raw_data': raw_movie,
            'error': str(e)
        })
        print(f"Validation failed for {raw_movie['raw_title']}: {e}")

print(f"\n{'='*60}")
print(f"Validation Results:")
print(f"Successfully validated: {len(validated_movies)} movies")
print(f"Failed validations: {len(failed_validations)} movies")
print(f"{'='*60}")

print("\nLast 15 validated movies:")
for i, movie in enumerate(validated_movies[-15:]):
    print(f"\n{i+1}. {movie.model_dump()}")


Validation Results:
Successfully validated: 139 movies
Failed validations: 0 movies

Last 15 validated movies:

1. {'title': 'Town & Country', 'year': 2001, 'budget_millions': 90.0, 'loss_millions': 10.4}

2. {'title': 'Transformers: The Last Knight', 'year': 2017, 'budget_millions': 238.5, 'loss_millions': 605.4}

3. {'title': 'Treasure Planet', 'year': 2002, 'budget_millions': 140.0, 'loss_millions': 109.6}

4. {'title': 'Tron: Ares', 'year': 2025, 'budget_millions': 220.0, 'loss_millions': 142.2}

5. {'title': 'Turning Red', 'year': 2022, 'budget_millions': 175.0, 'loss_millions': 19.8}

6. {'title': 'Valerian and the City of a Thousand Planets', 'year': 2017, 'budget_millions': 178.6, 'loss_millions': 225.9}

7. {'title': 'West Side Story', 'year': 2021, 'budget_millions': 100.0, 'loss_millions': 76.0}

8. {'title': 'Wild Wild West', 'year': 1999, 'budget_millions': 170.0, 'loss_millions': 217.7}

9. {'title': 'Windtalkers', 'year': 2002, 'budget_millions': 117.5, 'loss_millions':

## Task 3: Enrich with OMDb Data (4 Marks)

Query OMDb API for each movie to get:
- Plot
- Metascore
- IMDb Rating
- Director
- Language

Handle API failures (Response='False') or missing fields ('N/A') gracefully by storing them as None. Do not delete the row.

In [ ]:
OMDB_API_KEY = "94c9ebe8"
OMDB_BASE_URL = "http://www.omdbapi.com/"

def query_omdb(title: str, year: int) -> dict:

    def clean(v):
        return None if v in ("N/A", None) else v
    params = {
        "t": title,
        "y": year,
        "apikey": OMDB_API_KEY,
        "plot": "short"}

    try:
        response = requests.get(OMDB_BASE_URL, params=params, timeout=10)
        data = response.json()
    except Exception:
        data = {"Response": "False"}

    if data.get("Response") == "False":
        return {
            "plot": None,
            "metascore": None,
            "imdb_rating": None,
            "director": None,
            "language": None,
            "omdb_year": None
        }
    else:
      return {
        "plot": clean(data.get("Plot")),
        "metascore": int(data["Metascore"]) if clean(data.get("Metascore")) else None,
        "imdb_rating": float(data["imdbRating"]) if clean(data.get("imdbRating")) else None,
        "director": clean(data.get("Director")),
        "language": clean(data.get("Language")),
        "omdb_year": int(data["Year"][:4]) if clean(data.get("Year")) else None
    }


In [ ]:

enriched_data = []

for movie in validated_movies:
    omdb = query_omdb(movie.title, movie.year)
    enriched_data.append({
        "title": movie.title,
        "year": movie.year,
        "budget_millions": movie.budget_millions,
        "loss_millions": movie.loss_millions,
        **omdb
    })
    time.sleep(0.2)
print("Querying OMDb API...")

print(f"\nEnriched {len(enriched_data)} movies with OMDb data")

print("\nFirst enriched entry:")
print(enriched_data[0])

Querying OMDb API...

Enriched 139 movies with OMDb data

First enriched entry:
{'title': 'The 13th Warrior', 'year': 1999, 'budget_millions': 130.0, 'loss_millions': 61.7, 'plot': 'A man, having fallen in love with the wrong woman, is sent by the sultan himself on a diplomatic mission to a distant land as an ambassador. Stopping at a Viking village port to restock on supplies, he finds himself unwittingly em...', 'metascore': 42, 'imdb_rating': 6.6, 'director': 'John McTiernan', 'language': 'English, Latin, Swedish, Norse, Old, Danish, Arabic', 'omdb_year': 1999}


## Task 4: Data Consistency Check (2 Marks)

Compare Wikipedia year with OMDb year:
- "Verified": Years match (±1 year tolerance)
- "Mismatch": Years differ by >1
- "Not Found": OMDb returned no data

In [ ]:
def determine_match_status(wiki_year: int, omdb_year: Optional[int]) -> str:
    if omdb_year is None:
        return "Not Found"
    if abs(wiki_year - omdb_year) <= 1:
        return "Verified"
    return "Mismatch"

for item in enriched_data:
    item["match_status"] = determine_match_status(item["year"], item["omdb_year"])
df_temp = pd.DataFrame(enriched_data)
print("Match Status Distribution:")
print(df_temp['match_status'].value_counts())

mismatches = df_temp[df_temp['match_status'] == 'Mismatch']
if len(mismatches) > 0:
    print(f"\nSample Mismatches (showing up to 5):")
    print(mismatches[['title', 'year', 'omdb_year', 'match_status']].head())


Match Status Distribution:
match_status
Verified     137
Not Found      2
Name: count, dtype: int64


## Task 5: Final Dataset & Categorization (4 Marks)

Create final DataFrame with:
- Loss_Category based on estimated loss:
  - "Catastrophic": Loss ≥ \$100M

  - "Severe": Loss between \$50M and \$100M

  - "Moderate": Loss < \$50M
- Save to `box_office_failures.csv`

Required columns: Title, Year, Director, Language, Budget_Millions, Loss_Millions, Loss_Category, IMDb_Rating, Metascore, Match_Status

In [ ]:
def categorize_loss(loss_millions: float) -> str:
    if loss_millions >= 100:
        return "Catastrophic"
    elif loss_millions >= 50:
        return "Severe"
    else:
        return "Moderate"
    pass

df_final = pd.DataFrame(enriched_data)

df_final["Loss_Category"] = df_final["loss_millions"].apply(categorize_loss)
df_final = df_final.rename(columns={
    "title": "Title",
    "year": "Year",
    "director": "Director",
    "language": "Language",
    "budget_millions": "Budget_Millions",
    "loss_millions": "Loss_Millions",
    "imdb_rating": "IMDb_Rating",
    "metascore": "Metascore",
    "match_status": "Match_Status"
})

df_final = df_final[
    [
        "Title", "Year", "Director", "Language",
        "Budget_Millions", "Loss_Millions",
        "Loss_Category", "IMDb_Rating",
        "Metascore", "Match_Status"
    ]
]
print("Final Dataset Summary:")
print(f"Total movies: {len(df_final)}")
print(f"\nLoss Category Distribution:")
print(df_final['Loss_Category'].value_counts())
print(f"\nBasic Statistics:")
print(df_final[['Budget_Millions', 'Loss_Millions', 'IMDb_Rating', 'Metascore']].describe())

print(f"\nFirst 10 rows of final dataset:")


director_flop_analysis = df_final.groupby('Director').agg(
    Total_Loss_Millions=('Loss_Millions', 'sum')
).reset_index()

print(director_flop_analysis.sort_values(by='Total_Loss_Millions', ascending=False).head(10))



Final Dataset Summary:
Total movies: 139

Loss Category Distribution:
Loss_Category
Catastrophic    63
Moderate        52
Severe          24
Name: count, dtype: int64

Basic Statistics:
       Budget_Millions  Loss_Millions  IMDb_Rating   Metascore
count       139.000000     139.000000   137.000000  134.000000
mean        129.552518     115.758993     5.816058   46.529851
std          58.126686     108.118768     1.098276   15.359693
min          17.000000       1.100000     2.100000    9.000000
25%          81.250000      26.600000     5.400000   37.000000
50%         117.500000      80.800000     6.000000   46.000000
75%         175.000000     173.850000     6.600000   56.750000
max         326.000000     605.400000     8.000000   85.000000

First 10 rows of final dataset:
Loss_Millions    0.0
dtype: float64
Director Flop Analysis (Top 10 by Total Loss):
               Director  Total_Loss_Millions
48   Jaume Collet-Serra                613.9
74          Michael Bay                60

In [ ]:
output_path = 'box_office_failures.csv'
df_final.to_csv(output_path, index=False)
print(f"✓ Dataset saved to: {output_path}")
print(f"✓ Total records: {len(df_final)}")

✓ Dataset saved to: box_office_failures.csv
✓ Total records: 139
